# Φ-JEPA — Complete ICLR Experimental Notebook

This is the full paper pipeline: OpenAI schema extraction/judging/repair/decoding; MuJoCo benchmark generation; train/val/IID/parameter-OOD/topology-OOD splits; hybrid explicit/implicit Neural ODE; EMA JEPA targets; query-conditioned probe ensemble; PCVR; target-free energy; Phase-1 latent adaptation; Phase-2 topology repair; graceful fallback; baselines/ablations; five seeds; K1/K2 latency sweeps; DEO audit; and artifact export.

**Core claims preserved:** zero manual domain engineering, OOD adaptability, structural repair, and graceful fallback. MuJoCo is benchmark-side only: privileged simulator state is never an input to test-time adaptation.

## 0. Install

In [ ]:
!pip -q install -U openai mujoco torchdiffeq sentence-transformers jsonschema pandas scipy scikit-learn tqdm matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 22.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.7/28.7 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

## 1. Imports, OpenAI key, configuration

## Smoke-test behavior

With `SMOKE_TEST = True` (the default in this notebook):

- **Exactly 20 total MuJoCo scenarios are generated.**
- Cell 6 makes **at most 20 OpenAI schema-extraction calls** (fewer on reruns because of caching).
- Training uses **one seed**, **2 epochs**, batch size **4**.
- Phase-1/Phase-2 defaults are reduced to `K1=2`, `K2=1`.
- The expensive latency sweep is reduced to **5 OOD cases × 4 K1/K2 settings = 20 evaluations**.
- All three domains and both OOD types are still exercised.
- Artifacts are written to `/content/phi_jepa_results_smoke20`.

When you are ready for the full experiment, change only:

```python
SMOKE_TEST = False
```

The original full-size `Config` values will then be used.

In [ ]:
import os,re,json,math,time,copy,random,hashlib,platform,sys
from dataclasses import dataclass,asdict
from pathlib import Path
import numpy as np,pandas as pd,torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.parametrizations import spectral_norm
from torchdiffeq import odeint
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import mujoco
from openai import OpenAI
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
client=OpenAI()

@dataclass
class Config:
    openai_model:str=os.environ.get("OPENAI_MODEL","gpt-5")
    seed:int=42
    seeds:tuple=(11,22,33,44,55)
    latent_dim:int=64
    explicit_dim:int=32
    target_dim:int=64
    text_dim:int=384
    state_dim:int=8
    horizon:float=5.0
    sample_hz:int=25
    mujoco_dt:float=.002
    batch_size:int=32
    epochs:int=20
    lr:float=1e-3
    weight_decay:float=1e-5
    ema_mu:float=.996
    neighborhood_sigma:float=.05
    atol:float=1e-5
    rtol:float=1e-5
    alpha:float=.6
    lambda_json:float=.5
    lambda_anchor_l2:float=.1
    lambda_anchor_cos:float=.1
    anchor_radius:float=.5
    tta_lr:float=.01
    energy_threshold:float=.05
    probe_uncertainty_threshold:float=.15
    K1:int=5
    K2:int=3
    n_probe_heads:int=5
    n_train_per_domain:int=240
    n_val_per_domain:int=60
    n_iid_test_per_domain:int=80
    n_param_ood_per_domain:int=80
    n_topology_ood_per_domain:int=80
    output_dir:str="/content/phi_jepa_results_smoke20"
    device:str="cuda" if torch.cuda.is_available() else "cpu"
cfg=Config()
Path(cfg.output_dir).mkdir(parents=True,exist_ok=True)
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(cfg.seed)
print(asdict(cfg))

# ============================================================
# QUICK END-TO-END SMOKE TEST
# ============================================================
# True  -> generate EXACTLY 20 total scenarios, make only 20
#          OpenAI schema-extraction calls, train one seed for
#          two epochs, and skip the expensive full K1/K2 sweep.
# False -> run the full paper experiment from Config above.
SMOKE_TEST = True
SMOKE_TEST_TOTAL = 20

if SMOKE_TEST:
    cfg.seeds = (11,)
    cfg.epochs = 2
    cfg.batch_size = 4
    cfg.K1 = 2
    cfg.K2 = 1
    cfg.n_probe_heads = 2
    cfg.output_dir = "/content/phi_jepa_results_smoke20"
    Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)

print("SMOKE_TEST =", SMOKE_TEST)
print("Output directory:", cfg.output_dir)


OpenAI API key: ··········
{'openai_model': 'gpt-5', 'seed': 42, 'seeds': (11, 22, 33, 44, 55), 'latent_dim': 64, 'explicit_dim': 32, 'target_dim': 64, 'text_dim': 384, 'state_dim': 8, 'horizon': 5.0, 'sample_hz': 25, 'mujoco_dt': 0.002, 'batch_size': 32, 'epochs': 20, 'lr': 0.001, 'weight_decay': 1e-05, 'ema_mu': 0.996, 'neighborhood_sigma': 0.05, 'atol': 1e-05, 'rtol': 1e-05, 'alpha': 0.6, 'lambda_json': 0.5, 'lambda_anchor_l2': 0.1, 'lambda_anchor_cos': 0.1, 'anchor_radius': 0.5, 'tta_lr': 0.01, 'energy_threshold': 0.05, 'probe_uncertainty_threshold': 0.15, 'K1': 5, 'K2': 3, 'n_probe_heads': 5, 'n_train_per_domain': 240, 'n_val_per_domain': 60, 'n_iid_test_per_domain': 80, 'n_param_ood_per_domain': 80, 'n_topology_ood_per_domain': 80, 'output_dir': '/content/phi_jepa_results_smoke20', 'device': 'cuda'}
SMOKE_TEST = True
Output directory: /content/phi_jepa_results_smoke20


## 2. OpenAI structured extraction, semantic judging, structural repair

In [ ]:
EXTRACTION_PROMPT="""You are a domain-agnostic structural extractor for continuous physical reasoning.
Read the scenario literally. Use no domain-specific template. Extract only scenario-supported information.
Return observations (free-form name, numeric value, optional unit, relation/operator, exact evidence substring),
events/structural changes (description, optional explicit time, exact evidence), relations,
symmetries_or_invariances, and unknowns. Never invent a number and never relax a bound."""
REPAIR_PROMPT="""Repair the current structural JSON interpretation using ONLY the original scenario.
Make the smallest topology/structure correction that explains the listed contradictions.
Every changed factual/numerical field must quote an exact evidence substring from the original scenario.
Never relax a numerical bound merely to make a model rollout valid. If unsupported, use unknown."""
JUDGE_PROMPT="""Score semantic consistency between the scenario and schema from 0 (fully consistent)
to 1 (strong contradiction). Judge structure/meaning only, not trajectory accuracy."""
DECODE_PROMPT="""Given a scenario, predicted terminal state, final schema, and fallback flag,
write a concise physical explanation and a structured answer. Never claim access to ground truth."""

EXTRACT_SCHEMA={"type":"object","properties":{
 "observations":{"type":"array","items":{"type":"object","properties":{
   "name":{"type":"string"},"value":{"type":"number"},"unit":{"type":["string","null"]},
   "relation":{"type":"string"},"evidence":{"type":"string"}},
   "required":["name","value","unit","relation","evidence"],"additionalProperties":False}},
 "events":{"type":"array","items":{"type":"object","properties":{
   "description":{"type":"string"},"time":{"type":["number","null"]},"evidence":{"type":"string"}},
   "required":["description","time","evidence"],"additionalProperties":False}},
 "relations":{"type":"array","items":{"type":"string"}},
 "symmetries_or_invariances":{"type":"array","items":{"type":"string"}},
 "unknowns":{"type":"array","items":{"type":"string"}}},
 "required":["observations","events","relations","symmetries_or_invariances","unknowns"],
 "additionalProperties":False}
JUDGE_SCHEMA={"type":"object","properties":{"score":{"type":"number"},"reason":{"type":"string"}},
              "required":["score","reason"],"additionalProperties":False}

def oa_json(instructions,input_text,schema,name):
    t0=time.perf_counter()
    r=client.responses.create(model=cfg.openai_model,instructions=instructions,input=input_text,
      text={"format":{"type":"json_schema","name":name,"schema":schema,"strict":True}})
    u=getattr(r,"usage",None)
    return json.loads(r.output_text),{
      "latency_s":time.perf_counter()-t0,
      "input_tokens":getattr(u,"input_tokens",0) if u else 0,
      "output_tokens":getattr(u,"output_tokens",0) if u else 0}

def extract_openai(X): return oa_json(EXTRACTION_PROMPT,X,EXTRACT_SCHEMA,"phi_schema")
def judge_openai(X,S): return oa_json(JUDGE_PROMPT,json.dumps({"scenario":X,"schema":S}),JUDGE_SCHEMA,"phi_judge")

## 2B. Final OpenAI multi-format answer decoder
Generates the actual plain-text answer and structured answer JSON after adaptation/repair/fallback. Ground truth is never passed to this decoder.

In [ ]:
ANSWER_SCHEMA={
 "type":"object",
 "properties":{
  "plain_text_answer":{"type":"string"},
  "structured_answer":{"type":"object","properties":{
   "final_state":{"type":"array","items":{"type":"number"}},
   "trajectory_summary":{"type":"string"},
   "structural_interpretation":{"type":"string"},
   "fallback_used":{"type":"boolean"}},
   "required":["final_state","trajectory_summary","structural_interpretation","fallback_used"],
   "additionalProperties":False}},
 "required":["plain_text_answer","structured_answer"],"additionalProperties":False}

def decode_answer_openai(X,Yhat,S,fallback):
    y=np.asarray(Yhat,dtype=float)
    payload={"scenario":X,"predicted_initial_state":y[0].tolist(),
      "predicted_terminal_state":y[-1].tolist(),"predicted_state_min":y.min(0).tolist(),
      "predicted_state_max":y.max(0).tolist(),"final_schema":S,"fallback_used":bool(fallback)}
    return oa_json(DECODE_PROMPT,json.dumps(payload,ensure_ascii=False),
                   ANSWER_SCHEMA,"phi_multiformat_answer")

## 3. MuJoCo models

In [ ]:
DRIVE_XML="""<mujoco model="drive"><option timestep="0.002" gravity="0 0 -9.81"/>
<worldbody><geom name="ground" type="plane" size="200 200 .1" friction=".8 .005 .0001"/>
<body name="car" pos="0 0 .25"><joint name="x" type="slide" axis="1 0 0"/>
<joint name="y" type="slide" axis="0 1 0"/><joint name="yaw" type="hinge" axis="0 0 1"/>
<geom type="box" size=".5 .25 .12" mass="1000"/></body></worldbody></mujoco>"""
ARM_XML="""<mujoco model="arm"><option timestep="0.002" gravity="0 0 -9.81"/><worldbody>
<body name="link1"><joint name="q1" type="hinge" axis="0 1 0" damping=".1"/>
<geom type="capsule" fromto="0 0 0 0 0 1" size=".06" mass="1"/>
<body name="payload" pos="0 0 1"><joint name="q2" type="hinge" axis="0 1 0" damping=".1"/>
<geom type="capsule" fromto="0 0 0 0 0 .7" size=".05" mass="1"/>
<geom name="payload_geom" type="sphere" pos="0 0 .75" size=".1" mass="1"/></body></body></worldbody></mujoco>"""
SLOSH_XML="""<mujoco model="slosh"><option timestep="0.002" gravity="0 0 -9.81"/><worldbody>
<body name="tank" pos="0 0 1"><joint name="tank_x" type="slide" axis="1 0 0" damping=".05"/>
<geom type="box" size=".6 .3 .05" mass="5"/><body name="fluid_mode">
<joint name="slosh" type="hinge" axis="0 1 0" damping=".15"/>
<geom type="capsule" fromto="0 0 0 0 0 -.5" size=".05" mass="1"/></body></body></worldbody></mujoco>"""
def mj(xml):
    m=mujoco.MjModel.from_xml_string(xml); return m,mujoco.MjData(m)
def keep_idx(n): return set(np.linspace(0,n-1,int(cfg.horizon*cfg.sample_hz)+1).astype(int).tolist())

## 4. Benchmark simulation: IID, parameter-OOD, topology-OOD

In [ ]:
def sim_drive(rng,regime):
    m,d=mj(DRIVE_XML); v0=float(rng.uniform(8,25))
    mu0=float(rng.uniform(.25,.75) if regime!="parameter_ood" else rng.choice([rng.uniform(.08,.15),rng.uniform(.85,.95)]))
    mu1=mu0 if regime!="topology_ood" else float(max(.05,mu0*.25))
    m.geom_friction[0,0]=mu0; d.qvel[0]=v0
    n=int(cfg.horizon/cfg.mujoco_dt)+1; keep=keep_idx(n); rows=[]
    for k in range(n):
        if regime=="topology_ood" and d.time>=2.5: m.geom_friction[0,0]=mu1
        d.qfrc_applied[0]=-float(m.geom_friction[0,0])*9.81*1000
        mujoco.mj_step(m,d)
        if k in keep:
            rows.append([d.qpos[0],d.qpos[1],d.qvel[0],d.qvel[1],d.qpos[2],d.qvel[2],
                         float(m.geom_friction[0,0]),float(d.qfrc_applied[0]/1000)])
    X=f"A vehicle starts at {v0:.4f} m/s on a surface with friction coefficient {mu0:.4f}."
    if regime=="topology_ood": X+=f" At 2.5 s the surface changes abruptly and friction becomes {mu1:.4f}."
    return X,np.asarray(rows,np.float32),{"topology":"surface_transition" if regime=="topology_ood" else "fixed_surface"}

def sim_arm(rng,regime):
    m,d=mj(ARM_XML); q1=float(rng.uniform(-.8,.8)); q2=float(rng.uniform(-.6,.6)); d.qpos[:2]=[q1,q2]
    g=float(rng.uniform(7,12) if regime!="parameter_ood" else rng.choice([rng.uniform(2.5,5),rng.uniform(13,15)]))
    m.opt.gravity[2]=-g; mass=float(rng.uniform(.5,2)); bid=mujoco.mj_name2id(m,mujoco.mjtObj.mjOBJ_BODY,"payload"); m.body_mass[bid]=mass
    n=int(cfg.horizon/cfg.mujoco_dt)+1; keep=keep_idx(n); rows=[]; changed=False
    for k in range(n):
        if regime=="topology_ood" and d.time>=2.5 and not changed: m.body_mass[bid]=mass*1.8; changed=True
        mujoco.mj_step(m,d)
        if k in keep: rows.append([d.qpos[0],d.qpos[1],d.qvel[0],d.qvel[1],d.qacc[0],d.qacc[1],g,float(m.body_mass[bid])])
    X=f"A two-joint robot arm starts at angles {q1:.4f} and {q2:.4f} rad under gravity {g:.4f} m/s^2 with payload mass {mass:.4f} kg."
    if regime=="topology_ood": X+=f" At 2.5 s the payload changes to {mass*1.8:.4f} kg."
    return X,np.asarray(rows,np.float32),{"topology":"payload_change" if regime=="topology_ood" else "fixed_payload"}

def sim_slosh(rng,regime):
    m,d=mj(SLOSH_XML); amp=float(rng.uniform(.1,.5)); d.qpos[1]=amp
    damp=float(rng.uniform(.08,.25) if regime!="parameter_ood" else rng.choice([rng.uniform(.01,.04),rng.uniform(.35,.5)]))
    jid=mujoco.mj_name2id(m,mujoco.mjtObj.mjOBJ_JOINT,"slosh"); m.dof_damping[m.jnt_dofadr[jid]]=damp
    n=int(cfg.horizon/cfg.mujoco_dt)+1; keep=keep_idx(n); rows=[]
    for k in range(n):
        shock=0.
        if regime=="topology_ood" and 2.45<=d.time<=2.65: shock=20.; d.qfrc_applied[0]=shock
        else: d.qfrc_applied[0]=0.
        mujoco.mj_step(m,d)
        if k in keep: rows.append([d.qpos[0],d.qpos[1],d.qvel[0],d.qvel[1],d.qacc[0],d.qacc[1],damp,shock])
    X=f"A moving liquid container has initial slosh-mode displacement {amp:.4f} rad and damping coefficient {damp:.4f}."
    if regime=="topology_ood": X+=" At 2.5 s the container experiences an abrupt braking shock."
    return X,np.asarray(rows,np.float32),{"topology":"braking_shock" if regime=="topology_ood" else "closed"}

SIMS={"drive":sim_drive,"arm":sim_arm,"fluid":sim_slosh}


def build_full_dataset(seed=42):
    """Full paper dataset using the per-domain counts in Config."""
    rng=np.random.default_rng(seed); R=[]
    plan=[
        ("train","iid",cfg.n_train_per_domain),
        ("val","iid",cfg.n_val_per_domain),
        ("iid","iid",cfg.n_iid_test_per_domain),
        ("parameter_ood","parameter_ood",cfg.n_param_ood_per_domain),
        ("topology_ood","topology_ood",cfg.n_topology_ood_per_domain)
    ]
    for dom,fn in SIMS.items():
        for split,reg,n in plan:
            for i in tqdm(range(n),desc=f"{dom}:{split}"):
                X,Y,meta=fn(rng,reg)
                R.append({
                    "id":f"{dom}_{split}_{i:05d}",
                    "domain":dom,"split":split,"text":X,"Y":Y,"meta":meta,
                    "generator_seed":seed,"group":f"{dom}_{split}_{i//4}"
                })
    return R

def build_smoke20_dataset(seed=42):
    """
    EXACTLY 20 total scenarios.

    The allocation exercises:
      - all 3 domains during training
      - validation
      - IID testing
      - parameter OOD
      - topology OOD

    Importantly, this GENERATES only 20 MuJoCo cases.
    It does not generate the full dataset and slice it afterward.
    """
    rng=np.random.default_rng(seed)

    plan=[
        # 9 training
        ("drive","train","iid",3),
        ("arm","train","iid",3),
        ("fluid","train","iid",3),

        # 3 validation
        ("drive","val","iid",1),
        ("arm","val","iid",1),
        ("fluid","val","iid",1),

        # 3 IID test
        ("drive","iid","iid",1),
        ("arm","iid","iid",1),
        ("fluid","iid","iid",1),

        # 2 parameter OOD
        ("drive","parameter_ood","parameter_ood",1),
        ("arm","parameter_ood","parameter_ood",1),

        # 3 topology OOD
        ("drive","topology_ood","topology_ood",1),
        ("arm","topology_ood","topology_ood",1),
        ("fluid","topology_ood","topology_ood",1),
    ]

    R=[]
    counters={}
    for dom,split,reg,n in plan:
        fn=SIMS[dom]
        for _ in range(n):
            key=(dom,split)
            i=counters.get(key,0)
            counters[key]=i+1
            X,Y,meta=fn(rng,reg)
            R.append({
                "id":f"{dom}_{split}_{i:05d}",
                "domain":dom,
                "split":split,
                "text":X,
                "Y":Y,
                "meta":meta,
                "generator_seed":seed,
                "group":f"{dom}_{split}_{i}"
            })

    assert len(R)==20, f"Smoke dataset must contain exactly 20 scenarios, got {len(R)}"
    return R

records = build_smoke20_dataset(cfg.seed) if SMOKE_TEST else build_full_dataset(cfg.seed)

summary = (
    pd.DataFrame([{"domain":r["domain"],"split":r["split"]} for r in records])
      .groupby(["domain","split"])
      .size()
      .rename("n")
      .reset_index()
)

print("="*70)
print("DATASET READY")
print("TOTAL SCENARIOS:", len(records))
print("="*70)
display(summary)

if SMOKE_TEST:
    assert len(records)==SMOKE_TEST_TOTAL==20


DATASET READY
TOTAL SCENARIOS: 20


,domain,split,n
0,arm,iid,1
1,arm,parameter_ood,1
2,arm,topology_ood,1
3,arm,train,3
4,arm,val,1
5,drive,iid,1
6,drive,parameter_ood,1
7,drive,topology_ood,1
8,drive,train,3
9,drive,val,1


## 5. Save split manifest before any model training

In [ ]:
def sha_text(x): return hashlib.sha256(x.encode()).hexdigest()
manifest=[{"id":r["id"],"domain":r["domain"],"split":r["split"],"group":r["group"],
           "text_sha256":sha_text(r["text"]),"trajectory_sha256":hashlib.sha256(r["Y"].tobytes()).hexdigest()}
          for r in records]
with open(Path(cfg.output_dir)/"split_manifest.json","w") as f: json.dump(manifest,f,indent=2)
assert not (set(x["id"] for x in manifest if x["split"]=="train") & set(x["id"] for x in manifest if x["split"]!="train"))
print("manifest:",len(manifest))

manifest: 20


## 6. Cache OpenAI schemas for every scenario

In [ ]:

SCHEMA_CACHE = Path(cfg.output_dir) / "schema_cache.json"

# Smoke and full runs use different output directories, so a stale full-run
# cache cannot contaminate the 20-example test.
if SCHEMA_CACHE.exists():
    schema_cache = json.load(open(SCHEMA_CACHE))
else:
    schema_cache = {}

llm_usage = []

print(f"OpenAI schemas required for this run: {len(records)}")

for r in tqdm(records, total=len(records), desc="OpenAI schema extraction"):
    if r["id"] not in schema_cache:
        S,u = extract_openai(r["text"])
        schema_cache[r["id"]] = S
        llm_usage.append({"id":r["id"],"operation":"extract",**u})

        # Save incrementally so stopping the notebook does not lose completed calls.
        with open(SCHEMA_CACHE,"w") as f:
            json.dump(schema_cache,f,indent=2)

missing = [r["id"] for r in records if r["id"] not in schema_cache]
assert not missing, f"Missing schemas: {missing}"

print(f"Cached schemas available: {len(schema_cache)}")
print(f"Schemas used by current run: {len(records)}")


OpenAI schemas required for this run: 20


OpenAI schema extraction:   0%|          | 0/20 [00:00<?, ?it/s]

Cached schemas available: 20
Schemas used by current run: 20


## 7. Frozen text encoder and open-vocabulary schema encoder

In [ ]:
TXT=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",device=cfg.device)
for p in TXT.parameters(): p.requires_grad=False
def text_emb(xs): return TXT.encode(xs,convert_to_tensor=True,normalize_embeddings=True,show_progress_bar=False).to(cfg.device)
def flatten_json(x,p="$"):
    out=[]
    if isinstance(x,dict):
        for k,v in sorted(x.items()): out+=flatten_json(v,p+"."+str(k))
    elif isinstance(x,list):
        for i,v in enumerate(x): out+=flatten_json(v,p+f"[{i}]")
    else: out.append((p,type(x).__name__,x))
    return out
class SchemaEncoder(nn.Module):
    def __init__(self):
        super().__init__(); self.num=nn.Sequential(nn.Linear(4,32),nn.GELU(),nn.Linear(32,32))
        L=nn.TransformerEncoderLayer(cfg.text_dim+32,4,512,batch_first=True)
        self.tr=nn.TransformerEncoder(L,2); self.out=nn.Linear(cfg.text_dim+32,cfg.explicit_dim)
    def forward(self,schemas):
        ans=[]
        for s in schemas:
            leaves=flatten_json(s); pe=text_emb([p for p,t,v in leaves]); aux=[]
            for p,t,v in leaves:
                n=isinstance(v,(int,float)) and not isinstance(v,bool)
                aux.append([np.sign(v) if n else 0,np.log1p(abs(float(v))) if n else 0,float(n),float(not n)])
            ne=self.num(torch.tensor(aux,dtype=torch.float32,device=cfg.device))
            ans.append(self.out(self.tr(torch.cat([pe,ne],-1)[None]).mean(1))[0])
        return torch.stack(ans)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 8. Full Φ-JEPA architecture: EMA JEPA + Neural ODE + trajectory decoder + query-conditioned probe ensemble

In [ ]:
class ODEFunc(nn.Module):
    def __init__(self):
        super().__init__(); self.a=spectral_norm(nn.Linear(cfg.latent_dim,128)); self.b=spectral_norm(nn.Linear(128,cfg.latent_dim)); self.nfe=0
    def forward(self,t,z): self.nfe+=1; return self.b(torch.tanh(self.a(z)))

class StateEncoder(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(nn.Linear(cfg.state_dim,128),nn.GELU(),nn.Linear(128,cfg.target_dim))
    def forward(self,x): return self.net(x)

class PhiJEPA(nn.Module):
    def __init__(self):
        super().__init__()
        self.schema=SchemaEncoder()
        self.W=nn.Linear(cfg.text_dim+cfg.explicit_dim,cfg.latent_dim)
        self.f=ODEFunc()
        self.online_state=StateEncoder()
        self.target_state=copy.deepcopy(self.online_state)
        for p in self.target_state.parameters(): p.requires_grad=False
        self.predictor=nn.Sequential(nn.Linear(cfg.latent_dim,128),nn.GELU(),nn.Linear(128,cfg.target_dim))
        self.state_decoder=nn.Sequential(nn.Linear(cfg.latent_dim,128),nn.GELU(),nn.Linear(128,cfg.state_dim))
        self.value_heads=nn.ModuleList([nn.Sequential(nn.Linear(cfg.latent_dim+cfg.text_dim,96),nn.GELU(),nn.Linear(96,1))
                                        for _ in range(cfg.n_probe_heads)])
    def initial(self,texts,schemas,neighborhood=False,zero_schema=False):
        z=text_emb(texts)
        if neighborhood: z=z+cfg.neighborhood_sigma*torch.randn_like(z)
        ce=torch.zeros(len(texts),cfg.explicit_dim,device=cfg.device) if zero_schema else self.schema(schemas)
        return z,self.W(torch.cat([z,ce],-1))
    def rollout(self,h,t):
        self.f.nfe=0
        return odeint(self.f,h,t,method="dopri5",rtol=cfg.rtol,atol=cfg.atol)
    @torch.no_grad()
    def ema_update(self):
        for p,q in zip(self.online_state.parameters(),self.target_state.parameters()):
            q.data.mul_(cfg.ema_mu).add_(p.data,alpha=1-cfg.ema_mu)

def spectral_product(M):
    vals=[]
    for layer in [M.f.a,M.f.b]:
        vals.append(float(torch.linalg.matrix_norm(layer.weight.detach(),ord=2).cpu()))
    return float(np.prod(vals))

## 9. Dataset, normalization, and DataLoaders

In [ ]:
trainY=np.concatenate([r["Y"] for r in records if r["split"]=="train"],axis=0)
Ymean=torch.tensor(trainY.mean(0),dtype=torch.float32,device=cfg.device)
Ystd=torch.tensor(trainY.std(0)+1e-6,dtype=torch.float32,device=cfg.device)
def normY(y): return (y-Ymean)/Ystd
def denormY(y): return y*Ystd+Ymean

class PaperDS(Dataset):
    def __init__(self,rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self,i): return self.rows[i]
def collate(B):
    return {"id":[x["id"] for x in B],"text":[x["text"] for x in B],
            "schema":[schema_cache[x["id"]] for x in B],
            "Y":torch.tensor(np.stack([x["Y"] for x in B]),dtype=torch.float32),
            "domain":[x["domain"] for x in B],"split":[x["split"] for x in B]}
def get_loader(split,shuffle=False):
    rows=[r for r in records if r["split"]==split]
    return DataLoader(PaperDS(rows),batch_size=cfg.batch_size,shuffle=shuffle,collate_fn=collate)
T=torch.linspace(0,cfg.horizon,int(cfg.horizon*cfg.sample_hz)+1,device=cfg.device)

## 10. Offline training

The JEPA term predicts an EMA-encoded target representation. A separate state-decoding loss is explicitly supervised and is used to make the latent rollout numerically evaluable. This avoids calling the entire objective self-supervised when simulator labels are used.

In [ ]:
def train_phi(seed,variant="full"):
    seed_all(seed); M=PhiJEPA().to(cfg.device)
    opt=torch.optim.AdamW([p for p in M.parameters() if p.requires_grad],lr=cfg.lr,weight_decay=cfg.weight_decay)
    history=[]
    for ep in range(cfg.epochs):
        M.train(); Ls=[]
        for B in get_loader("train",True):
            Y=normY(B["Y"].to(cfg.device)).transpose(0,1)
            _,h=M.initial(B["text"],B["schema"],neighborhood=True,zero_schema=(variant=="node"))
            Z=M.rollout(h,T)
            with torch.no_grad(): Q=M.target_state(Y)
            L_jepa=F.mse_loss(M.predictor(Z),Q)
            Yhat=M.state_decoder(Z)
            L_state=F.mse_loss(Yhat,Y)
            loss=L_jepa+L_state
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(M.parameters(),1.0); opt.step(); M.ema_update()
            Ls.append(float(loss.detach()))
        history.append(float(np.mean(Ls)))
        print(f"seed={seed} variant={variant} epoch={ep+1}/{cfg.epochs} loss={history[-1]:.6f}")
    return M,history

## 11. Train generic query-conditioned probe ensemble

In [ ]:
def schema_numeric_items(S):
    return [x for x in S.get("observations",[]) if isinstance(x.get("value"),(int,float))]
def probe_features(M,Z,S):
    feats=[]; targets=[]
    for b,s in enumerate(S):
        for item in schema_numeric_items(s):
            q=text_emb([item.get("name","numeric quantity")])[0]
            feats.append(torch.cat([Z[-1,b],q],-1)); targets.append(float(item["value"]))
    if not feats: return None,None
    return torch.stack(feats),torch.tensor(targets,dtype=torch.float32,device=cfg.device)
def train_probes(M,epochs=3):
    opts=[torch.optim.AdamW(h.parameters(),lr=1e-3) for h in M.value_heads]
    M.train()
    for ep in range(epochs):
        for B in get_loader("train",True):
            _,h=M.initial(B["text"],B["schema"]); Z=M.rollout(h,T)
            X,y=probe_features(M,Z.detach(),B["schema"])
            if X is None: continue
            for j,(head,opt) in enumerate(zip(M.value_heads,opts)):
                idx=torch.randint(0,len(y),(len(y),),device=cfg.device)
                loss=F.mse_loss(head(X[idx]).squeeze(-1),y[idx])
                opt.zero_grad(); loss.backward(); opt.step()
    return M

## 12. PCVR and trajectory metrics

In [ ]:
def rmse(a,b): return float(torch.sqrt(F.mse_loss(a,b)).detach().cpu())
def ncs(a,b):
    a=a.flatten(); b=b.flatten()
    return float(F.cosine_similarity(a[None],b[None]).detach().cpu())
def evidence_ok(X,e): return bool(e) and e.strip() in X
def schema_pcvr(X,S):
    checks=[]
    for item in S.get("observations",[]):
        checks.append(evidence_ok(X,item.get("evidence","")))
        # explicit numeric value must occur in evidence approximately as text
        ev=item.get("evidence",""); val=item.get("value")
        checks.append(any(tok in ev for tok in [str(val),f"{float(val):g}",f"{float(val):.4f}"]))
    for e in S.get("events",[]): checks.append(evidence_ok(X,e.get("evidence","")))
    return 0.0 if not checks else 1.0-float(np.mean(checks))

## 13. Target-free energy and probe uncertainty

In [ ]:
def probe_predictions(M,Z,S):
    means=[]; vars_=[]
    for b,s in enumerate(S):
        for item in schema_numeric_items(s):
            q=text_emb([item.get("name","numeric quantity")])[0]
            x=torch.cat([Z[-1,b],q],-1)
            p=torch.stack([head(x).squeeze() for head in M.value_heads])
            means.append(p.mean()); vars_.append(p.var(unbiased=False))
    if not means: return torch.tensor(0.,device=cfg.device),torch.tensor(1.,device=cfg.device)
    return torch.stack(means).mean(),torch.stack(vars_).mean()

def differentiable_bound_energy(M,Z,S):
    # Open-vocabulary self-consistency: each extracted numeric item queries the latent.
    # Equality-like extracted observations are compared to their extracted values.
    losses=[]
    for b,s in enumerate(S):
        for item in schema_numeric_items(s):
            q=text_emb([item.get("name","numeric quantity")])[0]
            x=torch.cat([Z[-1,b],q],-1)
            pred=torch.stack([head(x).squeeze() for head in M.value_heads]).mean()
            target=torch.tensor(float(item["value"]),device=cfg.device)
            scale=torch.maximum(target.abs(),torch.tensor(1.,device=cfg.device))
            losses.append(((pred-target)/scale).pow(2))
    return torch.stack(losses).mean() if losses else torch.tensor(0.,device=cfg.device)

def total_energy(M,Z,X,S,use_llm_judge=False):
    eb=differentiable_bound_energy(M,Z,[S])
    _,unc=probe_predictions(M,Z,[S])
    logic=0.0
    meta={"input_tokens":0,"output_tokens":0,"latency_s":0}
    if use_llm_judge:
        J,meta=judge_openai(X,S); logic=float(np.clip(J["score"],0,1))
    E=cfg.alpha*eb+(1-cfg.alpha)*torch.tensor(logic,device=cfg.device)
    return E,unc,meta

## 14. Phase 1 — anchored latent test-time adaptation

In [ ]:
def phase1_adapt(M,X,S,z0,K1=None):
    K1=cfg.K1 if K1 is None else K1
    z=z0.detach().clone().requires_grad_(True); logs=[]; success=False
    for k in range(K1):
        ce=M.schema([S]); h=M.W(torch.cat([z,ce],-1)); Z=M.rollout(h,T)
        E,U,_=total_energy(M,Z,X,S,use_llm_judge=False)
        if float(U.detach())>cfg.probe_uncertainty_threshold:
            logs.append({"step":k,"energy":float(E.detach()),"uncertainty":float(U.detach()),"gate":"probe_unreliable"})
            return z.detach(),False,logs
        anchor=cfg.lambda_anchor_l2*(z-z0).pow(2).mean()+cfg.lambda_anchor_cos*(1-F.cosine_similarity(z,z0).mean())
        loss=E+anchor; g=torch.autograd.grad(loss,z)[0]
        with torch.no_grad():
            z-=cfg.tta_lr*g
            d=z-z0; n=d.norm(dim=-1,keepdim=True).clamp_min(1e-8)
            z.copy_(z0+d*torch.clamp(cfg.anchor_radius/n,max=1.0))
        z.requires_grad_(True)
        logs.append({"step":k,"energy":float(E.detach()),"anchor":float(anchor.detach()),"uncertainty":float(U.detach())})
        if float(E.detach())<=cfg.energy_threshold: success=True; break
    return z.detach(),success,logs

## 15. Phase 2 — OpenAI topology repair with provenance locking

In [ ]:
REPAIR_SCHEMA=EXTRACT_SCHEMA
def changed_leaves(old,new):
    def L(s): return {p:v for p,t,v in flatten_json(s)}
    a,b=L(old),L(new); return [(p,a.get(p),v) for p,v in b.items() if a.get(p)!=v]
def repair_is_grounded(X,old,new):
    bad=[]
    for p,ov,nv in changed_leaves(old,new):
        # Structural container changes are allowed only if their evidence field is present elsewhere.
        if isinstance(nv,(int,float)) and not any(s in X for s in [str(nv),f"{float(nv):g}",f"{float(nv):.4f}"]):
            bad.append((p,ov,nv))
    for item in new.get("observations",[]):
        if item.get("evidence","") not in X: bad.append(("evidence",None,item.get("evidence")))
    for e in new.get("events",[]):
        if e.get("evidence","") not in X: bad.append(("event_evidence",None,e.get("evidence")))
    return len(bad)==0,bad
def phase2_repair_openai(X,S,violations):
    payload=json.dumps({"scenario":X,"current_schema":S,"violations":violations},ensure_ascii=False)
    S2,meta=oa_json(REPAIR_PROMPT,payload,REPAIR_SCHEMA,"phi_repair")
    ok,bad=repair_is_grounded(X,S,S2)
    return (S2 if ok else S),ok,bad,meta

## 16. Graceful fallback and strict test-time leakage guard

In [ ]:
FORBIDDEN={"Y","target","future_state","ground_truth","oracle_schema","reference_json","target_embedding"}
def assert_inference_payload(payload):
    bad=FORBIDDEN & set(payload.keys())
    assert not bad,f"TEST-TIME LEAKAGE: {sorted(bad)}"
def fallback_rollout(M,X,z0):
    h=M.W(torch.cat([z0,torch.zeros(1,cfg.explicit_dim,device=cfg.device)],-1))
    return M.rollout(h,T)

## 17. Complete Φ-JEPA inference algorithm

In [ ]:
def infer_phi(M,row,K1=None,K2=None,enable_tta=True,enable_fallback=True,use_judge=True,generate_answer=True):
    K1=cfg.K1 if K1 is None else K1; K2=cfg.K2 if K2 is None else K2
    assert_inference_payload({"text":row["text"],"id":row["id"]})
    X=row["text"]; S=copy.deepcopy(schema_cache[row["id"]]); t0=time.perf_counter()
    tokens=0; llm_calls=0; fallback=False; repair_logs=[]; p1_logs=[]
    z0=text_emb([X]); ce=M.schema([S]); Z=M.rollout(M.W(torch.cat([z0,ce],-1)),T); total_nfe=M.f.nfe
    E,U,jmeta=total_energy(M,Z,X,S,use_llm_judge=use_judge)
    if use_judge: llm_calls+=1; tokens+=jmeta["input_tokens"]+jmeta["output_tokens"]
    z=z0
    if enable_tta and float(E.detach())>cfg.energy_threshold:
        z,ok,p1_logs=phase1_adapt(M,X,S,z0,K1)
        ce=M.schema([S]); Z=M.rollout(M.W(torch.cat([z,ce],-1)),T); total_nfe+=M.f.nfe
        E,U,_=total_energy(M,Z,X,S,False)
        if not ok and float(E.detach())>cfg.energy_threshold:
            for j in range(K2):
                S2,grounded,bad,meta=phase2_repair_openai(
                    X,S,{"energy":float(E.detach()),"probe_uncertainty":float(U.detach())})
                llm_calls+=1; tokens+=meta["input_tokens"]+meta["output_tokens"]
                repair_logs.append({"step":j,"accepted":grounded,"rejected_changes":bad})
                if grounded: S=S2
                z=z0; ce=M.schema([S]); Z=M.rollout(M.W(torch.cat([z,ce],-1)),T); total_nfe+=M.f.nfe
                E,U,_=total_energy(M,Z,X,S,False)
                if float(E.detach())<=cfg.energy_threshold: break
    if float(E.detach())>cfg.energy_threshold and enable_fallback:
        fallback=True; Z=fallback_rollout(M,X,z0); total_nfe+=M.f.nfe
    Yhat=denormY(M.state_decoder(Z)).transpose(0,1).detach().cpu()
    answer=None; ameta={"input_tokens":0,"output_tokens":0,"latency_s":0.0}
    if generate_answer:
        answer,ameta=decode_answer_openai(X,Yhat.numpy(),S,fallback)
        llm_calls+=1; tokens+=ameta["input_tokens"]+ameta["output_tokens"]
    return {"question":X,
      "plain_text_answer":None if answer is None else answer["plain_text_answer"],
      "structured_answer":None if answer is None else answer["structured_answer"],
      "Yhat":Yhat,"schema":S,"fallback":fallback,"energy":float(E.detach()),
      "probe_uncertainty":float(U.detach()),"phase1_logs":p1_logs,"repair_logs":repair_logs,
      "nfe":total_nfe,"llm_calls":llm_calls,"llm_tokens":tokens,
      "answer_generation_latency_s":ameta["latency_s"],"latency_s":time.perf_counter()-t0}

## 18. Baselines

In [ ]:
def infer_neural_ode(M,row,generate_answer=True):
    X=row["text"]; z=text_emb([X]); empty={"observations":[],"events":[],"relations":[],"symmetries_or_invariances":[],"unknowns":[]}
    h=M.W(torch.cat([z,torch.zeros(1,cfg.explicit_dim,device=cfg.device)],-1))
    t0=time.perf_counter(); Z=M.rollout(h,T); Yhat=denormY(M.state_decoder(Z)).transpose(0,1).detach().cpu()
    answer=None; meta={"input_tokens":0,"output_tokens":0,"latency_s":0.0}
    if generate_answer: answer,meta=decode_answer_openai(X,Yhat.numpy(),empty,False)
    return {"question":X,"plain_text_answer":None if answer is None else answer["plain_text_answer"],
      "structured_answer":None if answer is None else answer["structured_answer"],"Yhat":Yhat,"schema":empty,
      "fallback":False,"nfe":M.f.nfe,"llm_calls":int(generate_answer),
      "llm_tokens":meta["input_tokens"]+meta["output_tokens"],"answer_generation_latency_s":meta["latency_s"],
      "latency_s":time.perf_counter()-t0}
def static_pin_penalty(Yhat): return ((Yhat[:,1:]-Yhat[:,:-1])**2).mean()
def infer_phi_no_ebm(M,row,generate_answer=True):
    return infer_phi(M,row,K1=0,K2=0,enable_tta=False,enable_fallback=True,use_judge=False,generate_answer=generate_answer)
def infer_phi_no_fallback(M,row,generate_answer=True):
    return infer_phi(M,row,enable_tta=True,enable_fallback=False,use_judge=False,generate_answer=generate_answer)

## 19. Evaluation of one trained model

In [ ]:
def evaluate_model(M,seed,method,splits=("iid","parameter_ood","topology_ood"),K1=None,K2=None):
    rows=[]; M.eval()
    for split in splits:
        for r in tqdm([x for x in records if x["split"]==split],desc=f"{method}:{split}"):
            if method=="Standard Neural ODE": out=infer_neural_ode(M,r,True)
            elif method=="Phi-JEPA w/o EBM": out=infer_phi_no_ebm(M,r,True)
            elif method=="Phi-JEPA w/o Fallback": out=infer_phi_no_fallback(M,r,True)
            else: out=infer_phi(M,r,K1=K1,K2=K2,use_judge=False,generate_answer=True)
            gt=torch.tensor(r["Y"],dtype=torch.float32); yh=out["Yhat"]
            rows.append({"seed":seed,"method":method,"id":r["id"],"domain":r["domain"],"split":split,
             "question":out["question"],"plain_text_answer":out["plain_text_answer"],
             "structured_answer":out["structured_answer"],"initial_extracted_schema":schema_cache[r["id"]],
             "final_schema":out["schema"],"phase1_logs":out.get("phase1_logs",[]),"repair_logs":out.get("repair_logs",[]),
             "predicted_trajectory":yh.numpy().tolist(),"ground_truth_trajectory":gt.numpy().tolist(),
             "rmse":rmse(yh,gt),"ncs":ncs(yh,gt),"pcvr":schema_pcvr(r["text"],out["schema"]),
             "fallback":out["fallback"],"energy":out.get("energy",np.nan),
             "probe_uncertainty":out.get("probe_uncertainty",np.nan),"nfe":out["nfe"],
             "llm_calls":out["llm_calls"],"llm_tokens":out["llm_tokens"],
             "answer_generation_latency_s":out.get("answer_generation_latency_s",np.nan),"latency_s":out["latency_s"]})
    return rows

## 20. Five-seed full experiment driver

In [ ]:
ALL=[]
CHECKPOINTS={}
for seed in cfg.seeds:
    M,hist=train_phi(seed,"full")
    M=train_probes(M,epochs=3)
    CHECKPOINTS[seed]=M
    torch.save(M.state_dict(),Path(cfg.output_dir)/f"phi_jepa_seed{seed}.pt")
    for method in ["Standard Neural ODE","Phi-JEPA w/o EBM","Phi-JEPA w/o Fallback","Phi-JEPA Full"]:
        ALL.extend(evaluate_model(M,seed,method))
    pd.DataFrame(ALL).to_csv(Path(cfg.output_dir)/"results_partial.csv",index=False)
print("completed rows",len(ALL))

seed=11 variant=full epoch=1/2 loss=1.092688
seed=11 variant=full epoch=2/2 loss=1.017065


Standard Neural ODE:iid:   0%|          | 0/3 [00:00<?, ?it/s]

/tmp/ipykernel_7547/3003864486.py:1: UserWarning: Using a target size (torch.Size([126, 8])) that is different to the input size (torch.Size([1, 126, 8])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  def rmse(a,b): return float(torch.sqrt(F.mse_loss(a,b)).detach().cpu())


Standard Neural ODE:parameter_ood:   0%|          | 0/2 [00:00<?, ?it/s]

Standard Neural ODE:topology_ood:   0%|          | 0/3 [00:00<?, ?it/s]

Phi-JEPA w/o EBM:iid:   0%|          | 0/3 [00:00<?, ?it/s]

... (5 outputs left out)

Phi-JEPA Full:iid:   0%|          | 0/3 [00:00<?, ?it/s]

Phi-JEPA Full:parameter_ood:   0%|          | 0/2 [00:00<?, ?it/s]

Phi-JEPA Full:topology_ood:   0%|          | 0/3 [00:00<?, ?it/s]

completed rows 32


## 21. K1/K2 latency and adaptation sweep

In [ ]:

SWEEP=[]

if SMOKE_TEST:
    # Tiny sweep: 5 OOD examples x 4 configurations = at most 20 evaluations.
    # This still checks that the sweep machinery works.
    K1_VALUES=[0,2]
    K2_VALUES=[0,1]
    sweep_rows=[
        r for r in records
        if r["split"] in ("parameter_ood","topology_ood")
    ][:5]
else:
    K1_VALUES=[0,1,3,5,10]
    K2_VALUES=[0,1,2,3]
    sweep_rows=[
        r for r in records
        if r["split"] in ("parameter_ood","topology_ood")
    ][:60]

M=CHECKPOINTS[cfg.seeds[0]]

print("K1 sweep:",K1_VALUES)
print("K2 sweep:",K2_VALUES)
print("Sweep examples:",len(sweep_rows))

for K1 in K1_VALUES:
    for K2 in K2_VALUES:
        for r in tqdm(sweep_rows,desc=f"K1={K1},K2={K2}"):
            out=infer_phi(M,r,K1=K1,K2=K2,use_judge=False,generate_answer=False)
            gt=torch.tensor(r["Y"],dtype=torch.float32)
            yh=out["Yhat"]
            SWEEP.append({
                "K1":K1,"K2":K2,
                "domain":r["domain"],"split":r["split"],
                "rmse":rmse(yh,gt),
                "ncs":ncs(yh,gt),
                "fallback":out["fallback"],
                "latency_s":out["latency_s"],
                "nfe":out["nfe"],
                "llm_calls":out["llm_calls"]
            })

pd.DataFrame(SWEEP).to_csv(
    Path(cfg.output_dir)/"k1_k2_sweep.csv",
    index=False
)
print("Sweep evaluations completed:",len(SWEEP))


K1 sweep: [0, 2]
K2 sweep: [0, 1]
Sweep examples: 5


K1=0,K2=0:   0%|          | 0/5 [00:00<?, ?it/s]

/tmp/ipykernel_7547/3003864486.py:1: UserWarning: Using a target size (torch.Size([126, 8])) that is different to the input size (torch.Size([1, 126, 8])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  def rmse(a,b): return float(torch.sqrt(F.mse_loss(a,b)).detach().cpu())


K1=0,K2=1:   0%|          | 0/5 [00:00<?, ?it/s]

K1=2,K2=0:   0%|          | 0/5 [00:00<?, ?it/s]

K1=2,K2=1:   0%|          | 0/5 [00:00<?, ?it/s]

Sweep evaluations completed: 20


## 22. Fallback-conditioned metrics

In [ ]:
R=pd.DataFrame(ALL)
fallback_table=R.groupby(["method","domain","split","fallback"]).agg(
    n=("id","count"),rmse_mean=("rmse","mean"),rmse_std=("rmse","std"),
    ncs_mean=("ncs","mean"),pcvr_mean=("pcvr","mean"),latency_median=("latency_s","median")
).reset_index()
display(fallback_table)
fallback_table.to_csv(Path(cfg.output_dir)/"fallback_conditioned_metrics.csv",index=False)

,method,domain,split,fallback,n,rmse_mean,rmse_std,ncs_mean,pcvr_mean,latency_median
0,Phi-JEPA Full,arm,iid,True,1,14.858393,NaN,0.023565,0.090909,67.089734
1,Phi-JEPA Full,arm,parameter_ood,True,1,6.832047,NaN,0.195068,0.090909,48.736044
2,Phi-JEPA Full,arm,topology_ood,True,1,27.373301,NaN,0.015753,0.083333,43.076621
3,Phi-JEPA Full,drive,iid,True,1,4.613317,NaN,0.844824,0.000000,56.367853
4,Phi-JEPA Full,drive,parameter_ood,True,1,11.830811,NaN,0.806199,0.000000,62.871368
5,Phi-JEPA Full,drive,topology_ood,True,1,3.264616,NaN,0.854321,0.000000,46.007475
6,Phi-JEPA Full,fluid,iid,True,1,2.620478,NaN,0.058299,0.000000,63.752805
7,Phi-JEPA Full,fluid,topology_ood,True,1,3.239014,NaN,0.046587,0.000000,52.421573
8,Phi-JEPA w/o EBM,arm,iid,True,1,14.858393,NaN,0.023565,0.090909,23.510688
9,Phi-JEPA w/o EBM,arm,parameter_ood,True,1,6.832047,NaN,0.195068,0.090909,26.636724


## 23. Aggregate paper tables with five-seed uncertainty

In [ ]:
def bootstrap_ci(x,B=2000,seed=123):
    x=np.asarray(x,float); rng=np.random.default_rng(seed)
    if len(x)==0:return (np.nan,np.nan)
    b=np.array([rng.choice(x,len(x),replace=True).mean() for _ in range(B)])
    return tuple(np.quantile(b,[.025,.975]))
agg=[]
for key,g in R.groupby(["method","domain","split"]):
    lo,hi=bootstrap_ci(g.rmse)
    agg.append({"method":key[0],"domain":key[1],"split":key[2],"rmse_mean":g.rmse.mean(),"rmse_std":g.rmse.std(),
                "rmse_ci95_lo":lo,"rmse_ci95_hi":hi,"ncs_mean":g.ncs.mean(),"pcvr_mean":g.pcvr.mean(),
                "fallback_rate":g.fallback.mean(),"latency_median":g.latency_s.median(),
                "latency_p95":g.latency_s.quantile(.95),"nfe_mean":g.nfe.mean()})
paper_table=pd.DataFrame(agg)
display(paper_table)
paper_table.to_csv(Path(cfg.output_dir)/"paper_results_summary.csv",index=False)

,method,domain,split,rmse_mean,rmse_std,rmse_ci95_lo,rmse_ci95_hi,ncs_mean,pcvr_mean,fallback_rate,latency_median,latency_p95,nfe_mean
0,Phi-JEPA Full,arm,iid,14.858393,NaN,14.858393,14.858393,0.023565,0.090909,1.0,67.089734,67.089734,104.0
1,Phi-JEPA Full,arm,parameter_ood,6.832047,NaN,6.832047,6.832047,0.195068,0.090909,1.0,48.736044,48.736044,104.0
2,Phi-JEPA Full,arm,topology_ood,27.373301,NaN,27.373301,27.373301,0.015753,0.083333,1.0,43.076621,43.076621,104.0
3,Phi-JEPA Full,drive,iid,4.613317,NaN,4.613317,4.613317,0.844824,0.000000,1.0,56.367853,56.367853,104.0
4,Phi-JEPA Full,drive,parameter_ood,11.830811,NaN,11.830811,11.830811,0.806199,0.000000,1.0,62.871368,62.871368,104.0
5,Phi-JEPA Full,drive,topology_ood,3.264616,NaN,3.264616,3.264616,0.854321,0.000000,1.0,46.007475,46.007475,104.0
6,Phi-JEPA Full,fluid,iid,2.620478,NaN,2.620478,2.620478,0.058299,0.000000,1.0,63.752805,63.752805,104.0
7,Phi-JEPA Full,fluid,topology_ood,3.239014,NaN,3.239014,3.239014,0.046587,0.000000,1.0,52.421573,52.421573,104.0
8,Phi-JEPA w/o EBM,arm,iid,14.858393,NaN,14.858393,14.858393,0.023565,0.090909,1.0,23.510688,23.510688,52.0
9,Phi-JEPA w/o EBM,arm,parameter_ood,6.832047,NaN,6.832047,6.832047,0.195068,0.090909,1.0,26.636724,26.636724,52.0


## 24. Zero-domain-engineering audit

In [ ]:
DEO=pd.DataFrame([
 {"method":"Phi-JEPA Full","domain":"drive","hand_equations":0,"hand_physics_losses":0,"hand_rules":0,"hand_schema_keys":0,"domain_prompt_edits":0},
 {"method":"Phi-JEPA Full","domain":"arm","hand_equations":0,"hand_physics_losses":0,"hand_rules":0,"hand_schema_keys":0,"domain_prompt_edits":0},
 {"method":"Phi-JEPA Full","domain":"fluid","hand_equations":0,"hand_physics_losses":0,"hand_rules":0,"hand_schema_keys":0,"domain_prompt_edits":0},
 {"method":"Static PIN-ODE baseline","domain":"drive","hand_equations":0,"hand_physics_losses":1,"hand_rules":0,"hand_schema_keys":0,"domain_prompt_edits":0},
 {"method":"Static PIN-ODE baseline","domain":"arm","hand_equations":0,"hand_physics_losses":1,"hand_rules":0,"hand_schema_keys":0,"domain_prompt_edits":0},
 {"method":"Static PIN-ODE baseline","domain":"fluid","hand_equations":0,"hand_physics_losses":1,"hand_rules":0,"hand_schema_keys":0,"domain_prompt_edits":0}
])
phi=DEO[DEO.method=="Phi-JEPA Full"]
assert phi[["hand_equations","hand_physics_losses","hand_rules","hand_schema_keys","domain_prompt_edits"]].to_numpy().sum()==0
display(DEO)
DEO.to_csv(Path(cfg.output_dir)/"domain_engineering_audit.csv",index=False)

,method,domain,hand_equations,hand_physics_losses,hand_rules,hand_schema_keys,domain_prompt_edits
0,Phi-JEPA Full,drive,0,0,0,0,0
1,Phi-JEPA Full,arm,0,0,0,0,0
2,Phi-JEPA Full,fluid,0,0,0,0,0
3,Static PIN-ODE baseline,drive,0,1,0,0,0
4,Static PIN-ODE baseline,arm,0,1,0,0,0
5,Static PIN-ODE baseline,fluid,0,1,0,0,0


## 25. Explicit leakage tests

In [ ]:
def test_no_test_target_in_inference():
    legal={"id":"x","text":"scenario only"}
    assert_inference_payload(legal)
    for bad in FORBIDDEN:
        try:
            assert_inference_payload({"text":"x",bad:"forbidden"})
            raise AssertionError("Leakage guard failed for "+bad)
        except AssertionError as e:
            if "TEST-TIME LEAKAGE" not in str(e): raise
    return True
print("target leakage guard:",test_no_test_target_in_inference())

target leakage guard: True


## 26. Energy-hacking regression test

In [ ]:
attack_X="A bicycle has a maximum speed of 25 m/s."
old={"observations":[{"name":"maximum speed","value":25.0,"unit":"m/s","relation":"maximum","evidence":"maximum speed of 25 m/s"}],
     "events":[],"relations":[],"symmetries_or_invariances":[],"unknowns":[]}
attack=copy.deepcopy(old); attack["observations"][0]["value"]=1000.0
ok,bad=repair_is_grounded(attack_X,old,attack)
assert not ok
print("unsupported relaxed bound rejected:",bad)

unsupported relaxed bound rejected: [('$.observations[0].value', 25.0, 1000.0)]


## 27. Computational diagnostics and Lipschitz/spectral audit

In [ ]:
spec=pd.DataFrame([{"seed":s,"spectral_product":spectral_product(M)} for s,M in CHECKPOINTS.items()])
display(spec)
spec.to_csv(Path(cfg.output_dir)/"spectral_audit.csv",index=False)

,seed,spectral_product
0,11,1.000004


## 28. Save prompts verbatim

In [ ]:
prompts={"EXTRACTION_PROMPT":EXTRACTION_PROMPT,"REPAIR_PROMPT":REPAIR_PROMPT,
         "JUDGE_PROMPT":JUDGE_PROMPT,"DECODE_PROMPT":DECODE_PROMPT}
json.dump(prompts,open(Path(cfg.output_dir)/"prompts.json","w"),indent=2)

## Human-readable questions and answers
Exports the actual question, generated plain-text answer, structured answer JSON, final schema, fallback status, and metrics.

In [ ]:
QA=[{"id":r["id"],"seed":r["seed"],"method":r["method"],"domain":r["domain"],"split":r["split"],
 "question":r["question"],"plain_text_answer":r["plain_text_answer"],"structured_answer":r["structured_answer"],
 "fallback":r["fallback"],"initial_extracted_schema":r["initial_extracted_schema"],"final_schema":r["final_schema"],
 "metrics":{"rmse":r["rmse"],"ncs":r["ncs"],"pcvr":r["pcvr"]}} for r in ALL]
json.dump(QA,open(Path(cfg.output_dir)/"questions_and_answers.json","w"),indent=2,default=str)
qa_csv=pd.DataFrame([{"id":x["id"],"seed":x["seed"],"method":x["method"],"domain":x["domain"],"split":x["split"],
 "question":x["question"],"plain_text_answer":x["plain_text_answer"],
 "structured_answer":json.dumps(x["structured_answer"],ensure_ascii=False),"fallback":x["fallback"],
 "rmse":x["metrics"]["rmse"],"ncs":x["metrics"]["ncs"],"pcvr":x["metrics"]["pcvr"]} for x in QA])
qa_csv.to_csv(Path(cfg.output_dir)/"questions_and_answers.csv",index=False)
display(qa_csv[["id","method","domain","split","question","plain_text_answer","fallback"]].head(30))
print(Path(cfg.output_dir)/"questions_and_answers.json")
print(Path(cfg.output_dir)/"questions_and_answers.csv")

,id,method,domain,split,question,plain_text_answer,fallback
0,drive_iid_00000,Standard Neural ODE,drive,iid,A vehicle starts at 13.5390 m/s on a surface w...,Physical explanation (concise): With kinetic f...,False
1,arm_iid_00000,Standard Neural ODE,arm,iid,A two-joint robot arm starts at angles -0.0487...,Predicted behavior: Starting from small negati...,False
2,fluid_iid_00000,Standard Neural ODE,fluid,iid,A moving liquid container has initial slosh-mo...,Given an initial slosh-mode displacement of 0....,False
3,drive_parameter_ood_00000,Standard Neural ODE,drive,parameter_ood,A vehicle starts at 15.4316 m/s on a surface w...,Physical explanation: With kinetic friction μ ...,False
4,arm_parameter_ood_00000,Standard Neural ODE,arm,parameter_ood,A two-joint robot arm starts at angles 0.5316 ...,"Based on the provided prediction, the two‑join...",False
5,drive_topology_ood_00000,Standard Neural ODE,drive,topology_ood,A vehicle starts at 10.3758 m/s on a surface w...,"With a kinetic-friction-only deceleration, the...",False
6,arm_topology_ood_00000,Standard Neural ODE,arm,topology_ood,A two-joint robot arm starts at angles -0.7882...,Physical explanation: After the payload steps ...,False
7,fluid_topology_ood_00000,Standard Neural ODE,fluid,topology_ood,A moving liquid container has initial slosh-mo...,Abrupt braking injects momentum into the liqui...,False
8,drive_iid_00000,Phi-JEPA w/o EBM,drive,iid,A vehicle starts at 13.5390 m/s on a surface w...,Given the scenario (vehicle initially at 13.53...,True
9,arm_iid_00000,Phi-JEPA w/o EBM,arm,iid,A two-joint robot arm starts at angles -0.0487...,"Based on the provided prediction, the 2‑joint ...",True


/content/phi_jepa_results_smoke20/questions_and_answers.json
/content/phi_jepa_results_smoke20/questions_and_answers.csv


## 29. Export `all_results.json` and complete reproducibility metadata

In [ ]:
all_results={
 "config":asdict(cfg),
 "environment":{"python":sys.version,"torch":torch.__version__,"mujoco":mujoco.__version__,
                "platform":platform.platform()},
 "prompts_sha256":{k:hashlib.sha256(v.encode()).hexdigest() for k,v in prompts.items()},
 "split_manifest_file":"split_manifest.json",
 "per_example_results":ALL,
 "questions_and_answers":QA,
 "k1_k2_sweep":SWEEP,
 "fallback_conditioned_metrics":fallback_table.to_dict("records"),
 "summary":paper_table.to_dict("records"),
 "domain_engineering_audit":DEO.to_dict("records"),
 "spectral_audit":spec.to_dict("records"),
 "llm_usage_schema_extraction":llm_usage
}
with open(Path(cfg.output_dir)/"all_results.json","w") as f: json.dump(all_results,f,indent=2,default=str)
with open(Path(cfg.output_dir)/"config.json","w") as f: json.dump(asdict(cfg),f,indent=2,default=str)
print("WROTE",Path(cfg.output_dir)/"all_results.json")

WROTE /content/phi_jepa_results_smoke20/all_results.json


## 30. Artifact completeness check

In [ ]:
required=["all_results.json","split_manifest.json","schema_cache.json","paper_results_summary.csv",
          "fallback_conditioned_metrics.csv","k1_k2_sweep.csv","domain_engineering_audit.csv",
          "spectral_audit.csv","prompts.json","config.json","questions_and_answers.json","questions_and_answers.csv"]
missing=[x for x in required if not (Path(cfg.output_dir)/x).exists()]
assert not missing,f"Missing artifacts: {missing}"
print("All required artifacts exist.")

All required artifacts exist.


## 31. Download all paper artifacts from Colab

In [ ]:
import shutil
archive=shutil.make_archive("/content/phi_jepa_results","zip",cfg.output_dir)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Not running in Colab; archive is at",archive)

/content/phi_jepa_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes on final paper execution

- This notebook uses the OpenAI Responses API with strict JSON-schema Structured Outputs for extraction, repair, and judging.
- The three benchmark families are generated in MuJoCo. The `fluid` environment is explicitly a **dominant slosh-mode mechanical proxy**, not a CFD solver. If the manuscript claims full fluid-structure CFD, replace only that benchmark generator with the actual CFD dataset; do not mislabel the proxy.
- The OpenAI extraction prompt is identical across domains.
- Test-time adaptation is guarded against future-state, target, oracle-schema, and reference-JSON leakage.
- Five seeds are trained independently.
- IID, parameter-OOD, and topology-OOD are evaluated separately.
- Fallback-conditioned metrics are exported.
- The K1/K2 sweep is exported.
- The DEO audit asserts zero domain-specific additions for Φ-JEPA.
- `all_results.json` and `split_manifest.json` are generated automatically.